In [1]:
from collections import Counter
import warnings
from pprint import pprint

import numpy as np

# Access astronomical databases
from pyvo import registry, io

# Moc and HEALPix tools
# from mocpy import MOC

# Coordinates manipulation
# from astropy.coordinates import SkyCoord

# Sky visualization
# from ipyaladin import Aladin  # version >=0.4.0

# For plots
# import matplotlib.pyplot as plt

# Partial description of VizieR example workflow

It is a generic notebook, highlighting what can be done once you chose a catalog. This workflow is suggested by [CDS](https://cdsweb.unistra.fr/) (Strasbourg Astronomical Data Center, house of [VizieR](https://vizier.cds.unistra.fr/viz-bin/VizieR)).

The notebook exploits [pyVO](https://pyvo.readthedocs.io/en/latest/), an advanced library  of The [Virtual Observatory](https://ivoa.net/).

[Astroquery](https://astroquery.readthedocs.io/en/latest/vizier/vizier.html) (not used here) is a well-documented, user-friendly alternative.

## 1. Setup

This example notebook has the following dependencies: 

**Required**
- pyvo : this library facilitates the access to the Virtual Observatory (VO) resources. VizieR is part of the VO.
This notebook needs version >=1.4.1
**Optional, for visualization**
- ipyaladin : this is the Aladin-lite sky viewer, bundled as a jupyter widget. It allows to plot catalogs and multi-order coverages (MOC)
- matplotlib : an other option to see catalog points and MOCs

## 2. Metadata exploration with the Virtual Observatory registry

This part uses [pyvo](https://pyvo.readthedocs.io/en) to connect to the VO registry.

In [2]:
# the catalogue name in VizieR: The Almagest Ptolemy's star catalogue
CATALOGUE = "V/61"

We first retrieve the catalogue information.

In [3]:
# each resource in the VO has an identifier, called ivoid. For vizier catalogs,
# the VO ids can be constructed like this:
catalogue_ivoid = f"ivo://CDS.VizieR/{CATALOGUE}"
# the actual query to the registry
voresource = registry.search(ivoid=catalogue_ivoid)[0]

In [4]:
# We can print metadata information about the catalogue
voresource.describe(verbose=True)

Almagest
Short Name: V/61
IVOA Identifier: ivo://cds.vizier/v/61
Access modes: tap#aux, web
- tap#aux: https://tapvizier.cds.unistra.fr/TAPVizieR/tap
- webpage: https://vizier.cds.unistra.fr/viz-bin/VizieR-2?-source=V/61

(no description available)

Waveband Coverage:

Source: 1987BICDS..33..125J
Authors: Ptolemy C.: Almagest (years 127-141)Manitius K.: 1913
More info: https://cdsarc.cds.unistra.fr/viz-bin/cat/V/61


We can also inspect in details the `resource` object and access the attributes not provided by the describe method. See for example, the first author of a resource: 

In [5]:
voresource.creators[0]

'Ptolemy C.: Almagest (years 127-141)Manitius K.: 1913'

## 3. Access the tabular data of this catalog

We can have a look at the tables available in the catalogue.

In [6]:
warnings.filterwarnings("ignore", category=io.vosi.vodataservice.W02)
tables = voresource.get_tables()
print(f"In this catalogue, we have {len(tables)} tables.")
for table_name, table in tables.items():
    print(f"{table_name}: {table.description}")

In this catalogue, we have 4 tables.
V/61/zod-n: Zodiacal stars, North
V/61/zod-s: Zodiacal stars, South
V/61/north: Northern part (except Zodiacal stars)
V/61/south: Southern part (except Zodiacal stars)


In [7]:
# We can also extract the tables names for later use
tables_names = list(tables.keys())
tables_names

['V/61/zod-n', 'V/61/zod-s', 'V/61/north', 'V/61/south']

The actual data can then be accessed using any of the ``access_modes`` of the voresource.

The web access is found by following the ``reference_url``

In [8]:
pprint(tables["V/61/zod-n"].columns)
print(voresource.access_modes())
print(voresource.reference_url)

[<BaseParam name="recno"/>,
 <BaseParam name="const"/>,
 <BaseParam name="hr"/>,
 <BaseParam name="m_hr"/>,
 <BaseParam name="seq"/>,
 <BaseParam name="lon"/>,
 <BaseParam name="lat-"/>,
 <BaseParam name="lat"/>,
 <BaseParam name="mag"/>,
 <BaseParam name="disag"/>]
{'tap#aux', 'web'}
https://cdsarc.cds.unistra.fr/viz-bin/cat/V/61


### 3.1 Execute a SQL/ADQL query

The ``tap#aux`` in the ``access_mode`` response indicates that we can also do a SQL/ADQL query for these VizieR tables.

On the first table of the catalogue, we execute an <a href='https://www.ivoa.net/documents/latest/ADQL.html'>ADQL</a> query.

In [9]:
# get the first table of the catalogue
first_table_name = tables_names[0]

# execute a synchronous ADQL query
tap_service = voresource.get_service("tap")
tap_records = tap_service.search(
    f'select ALL * from "{first_table_name}"',
)
tap_records

<DALResultsTable length=167>
recno Const    HR   m_HR   Seq  ... Lat-         Lat           Mag   Disag
                                ...                            mag        
int32 object int16 object int16 ... str1       float64       float64  str1
----- ------ ----- ------ ----- ... ---- ------------------- ------- -----
    1  ARIES   545            1 ...    +   7.333333333333332     3.3      
    2  ARIES   553            2 ...    +   8.333333333333332     3.0      
    3  ARIES   646            3 ...    +   7.666666666666665     5.0      
    4  ARIES   669            4 ...    +   5.999999999999999     5.0      
    5  ARIES   563            5 ...    +   5.833333333333332     5.0      
    6  ARIES   773            6 ...    +   5.999999999999999     6.0      
    7  ARIES   887     /8     7 ...    +   4.833333333333333     5.0      
    8  ARIES   951            8 ...    +  1.6666666666666663     4.0      
    9  ARIES   972            9 ...    +  2.4999999999999996     4.0   

In [10]:
all_const = {}
all_stars = 0
all_objects_hr = []
for table_name in tables_names:
    tap_records = tap_service.search(
    f'select ALL * from "{table_name}"',
)
    print(f"{table_name}:")
    print(f"Number of records: {len(tap_records)}")
    cntr = Counter(tap_records["Const"])
    all_objects_hr.extend(tap_records["HR"])
    all_const.update(dict(cntr))
    all_stars += cntr.total()
    print(all_const)

# all_objects_not_in_hr = [item for item in all_objects_hr if not isinstance(item, np.int16) or item == 0]
all_stars_in_hr = [item for item in all_objects_hr if isinstance(item, np.int16) and item != 0]
print(f"Number of constellations: {len(all_const)}")
print(f"Number of objects: {all_stars}")
print(f"Number of stars identified in the Bright Star Catalogue: {len(all_stars_in_hr)}")

# Sort by value
sorted_items = sorted(all_const.items(), key=lambda item: item[1], reverse=True)
print()
print("All constellations:")
for i, item in enumerate(sorted_items):
    print(item, end=' ')
    if (i + 1) % 3 == 0:
        print()


V/61/zod-n:
Number of records: 167
{'ARIES': 18, 'TAURUS': 44, 'GEMINI': 25, 'CANCER': 13, 'LEO': 35, 'VIRGO': 32}
V/61/zod-s:
Number of records: 183
{'ARIES': 18, 'TAURUS': 44, 'GEMINI': 25, 'CANCER': 13, 'LEO': 35, 'VIRGO': 32, 'LIBRA': 17, 'SCORPIUS': 24, 'SAGITTARIUS': 31, 'CAPRICORNUS': 28, 'AQUARIUS': 45, 'PISCES': 38}
V/61/north:
Number of records: 360
{'ARIES': 18, 'TAURUS': 44, 'GEMINI': 25, 'CANCER': 13, 'LEO': 35, 'VIRGO': 32, 'LIBRA': 17, 'SCORPIUS': 24, 'SAGITTARIUS': 31, 'CAPRICORNUS': 28, 'AQUARIUS': 45, 'PISCES': 38, 'URSA MINOR': 8, 'URSA MAJOR': 35, 'DRACO': 31, 'CEPHEUS': 13, 'BOOTES': 23, 'CORONA BOREALIS': 8, 'HERCULES': 29, 'LYRA': 10, 'CYGNUS': 19, 'CASSIOPEIA': 13, 'PERSEUS': 29, 'AURIGA': 14, 'OPHIUCHUS': 29, 'SERPENS': 18, 'SAGITTA': 5, 'AQUILA': 15, 'DELPHINUS': 10, 'EQUULEUS': 4, 'PEGASUS': 20, 'ANDROMEDA': 23, 'TRIANGULUM': 4}
V/61/south:
Number of records: 317
{'ARIES': 18, 'TAURUS': 44, 'GEMINI': 25, 'CANCER': 13, 'LEO': 35, 'VIRGO': 32, 'LIBRA': 17, 'SCO